# Pydantic AI search diagnostics

This notebook intentionally ignores the project prompts. It checks the real chain in three steps:

1. model returns a plain answer without tools;
2. model may call the real `search_spots` tool and returns plain text;
3. model must call the real `search_spots` tool and return a structured Pydantic object.


In [ ]:
import asyncio
import os
from pathlib import Path
from pprint import pprint

from pydantic import BaseModel, Field
from pydantic_ai import Agent

from capabilities.search import SearchCapability
from core.models import StargazingSpot


async def run_with_timeout(label: str, awaitable, timeout: float = 45):
    print(f"START: {label}")
    try:
        result = await asyncio.wait_for(awaitable, timeout=timeout)
    except TimeoutError:
        print(f"TIMEOUT after {timeout}s: {label}")
        raise
    except Exception as exc:
        print(f"ERROR in {label}: {type(exc).__name__}: {exc}")
        raise
    else:
        print(f"DONE: {label}")
        return result


def load_dotenv(path: str = ".env") -> None:
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


load_dotenv()

MODEL = os.getenv(
    "LAZY_STELLAR_MODEL",
    "openrouter:nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
)

print(f"MODEL: {MODEL}")
for key in ["OPENROUTER_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'missing'}")


In [ ]:
class SpotSearchReport(BaseModel):
    spots_found: int = Field(description="Number of spots returned by the search tool.")
    summary: str = Field(description="Concise user-facing answer in English.")
    spots: list[StargazingSpot] = Field(default_factory=list)


## 1. Plain model call, no tools

If this hangs or fails, the problem is model/provider configuration, not search or structured output.


In [ ]:
plain_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    instructions="Reply with one short sentence. Do not use tools.",
)

plain_result = await run_with_timeout(
    "plain model call",
    plain_agent.run("Say hello and name one reason dark skies matter."),
)

print("OUTPUT:")
print(plain_result.output)
print("\nUSAGE:")
print(plain_result.usage)


## 2. Real search tool, plain text output

If step 1 works but this fails, inspect whether the model called `search_spots`, whether Tavily/DuckDuckGo failed, or whether the tool result came back as an error string.


In [ ]:
search_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    capabilities=[SearchCapability()],
    instructions=(
        "You are a stargazing assistant. For location-specific recommendations, "
        "call search_spots exactly once before answering. Then summarize the returned spots. "
        "If the tool returns an error string, report that exact failure briefly."
    ),
)

search_text_result = await run_with_timeout(
    "search tool + plain text output",
    search_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("OUTPUT:")
print(search_text_result.output)
print("\nUSAGE:")
print(search_text_result.usage)
print("\nMESSAGES:")
for message in search_text_result.all_messages():
    print(type(message).__name__, message)


## 3. Real search tool, structured Pydantic output

If steps 1 and 2 work but this fails, the problem is structured output/tool interaction. This cell forces a `SpotSearchReport` final result.


In [ ]:
structured_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    output_type=SpotSearchReport,
    capabilities=[SearchCapability()],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Use the tool result to fill SpotSearchReport. "
        "If search_spots returns an error string, return spots_found=0, spots=[], and put the error in summary. "
        "Do not invent spots that were not returned by the tool."
    ),
)

structured_result = await run_with_timeout(
    "search tool + structured Pydantic output",
    structured_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("PYDANTIC OUTPUT:")
pprint(structured_result.output.model_dump())
print("\nJSON:")
print(structured_result.output.model_dump_json(indent=2))
print("\nUSAGE:")
print(structured_result.usage)
print("\nMESSAGES:")
for message in structured_result.all_messages():
    print(type(message).__name__, message)
